In [ ]:
# Student distillation (Colab): clone repo, install deps, device
import sys
from pathlib import Path

REPO_URL = "https://github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git"
PROJECT_DIR = "/content/Secure-Inference-Token-Reduced-VIT"

if Path(PROJECT_DIR).exists():
    %cd $PROJECT_DIR
    !git pull
else:
    !git clone $REPO_URL $PROJECT_DIR
    %cd $PROJECT_DIR
!pip install -q -r $PROJECT_DIR/requirements.txt

sys.path.insert(0, PROJECT_DIR)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Force Python to reload code (no old .pyc)
import importlib
import sys
for k in list(sys.modules.keys()):
    if k.startswith("models") or k.startswith("training") or k.startswith("data"):
        del sys.modules[k]
!find $PROJECT_DIR -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null; echo "cache cleared"

In [ ]:
# Mount Google Drive (upload teacher_best.pt there if not in repo)
from google.colab import drive
drive.mount("/content/drive")

# Path to teacher checkpoint: Drive or local
TEACHER_CKPT = "/content/drive/MyDrive/Secure-Inference-Token-Reduced-VIT/checkpoints/teacher_best.pt"

In [ ]:
from data import get_lc25000_root, get_dataloaders

root = get_lc25000_root()
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=root, batch_size=32, val_ratio=0.15, test_ratio=0.15, seed=42,
    subdir_depth=2, image_size=224, num_workers=2,
)
ds = train_loader.dataset
num_classes = ds.num_classes
print("Classes:", ds.class_names)
print("Train:", len(ds), "Val:", len(val_loader.dataset), "Test:", len(test_loader.dataset))

In [ ]:
from training import load_teacher_checkpoint, train_student
from models import get_student_vit

if not Path(TEACHER_CKPT).exists():
    raise FileNotFoundError(
        f"Teacher checkpoint not found: {TEACHER_CKPT}\n"
        "Upload teacher_best.pt to this path on Google Drive, or set TEACHER_CKPT in cell 1."
    )
teacher = load_teacher_checkpoint(TEACHER_CKPT, device)
teacher.eval()

# Student config (save for plots)
NUM_OUTPUT_TOKENS = 97
EMBED_DIM = 384
DEPTH = 6
NUM_HEADS = 6
TEMPERATURE = 4.0
ALPHA = 0.0
USE_HARD_LABELS = True
LR = 1e-5  # safer default for bounded (non-softmax) student

student = get_student_vit(
    num_classes=num_classes,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    num_output_tokens=NUM_OUTPUT_TOKENS,
    norm_mode="layernorm",
)

run_config = {
    "num_output_tokens": NUM_OUTPUT_TOKENS,
    "embed_dim": EMBED_DIM,
    "depth": DEPTH,
    "num_heads": NUM_HEADS,
    "norm_mode": "layernorm",
    "temperature": TEMPERATURE,
    "alpha": ALPHA,
    "use_hard_labels": USE_HARD_LABELS,
    "lr": LR,
}
print("Student and config ready.")

In [ ]:
# Distillation training; logs and best checkpoint saved for later plots
LOG_DIR = f"{PROJECT_DIR}/logs"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
log_path = f"{LOG_DIR}/student_K{NUM_OUTPUT_TOKENS}.csv"
checkpoint_dir = CKPT_DIR

student = train_student(
    student, teacher, train_loader, val_loader, device,
    epochs=30, lr=LR,
    temperature=TEMPERATURE, alpha=ALPHA, use_hard_labels=USE_HARD_LABELS,
    log_path=log_path,
    checkpoint_dir=checkpoint_dir,
    run_config=run_config,
)
print("Training done. Check", LOG_DIR, "and", CKPT_DIR)

In [ ]:
# Optional: evaluate best student on test set and save results (for plots)
from training import load_student_checkpoint, evaluate_teacher

student_ckpt = f"{CKPT_DIR}/student_best.pt"
student = load_student_checkpoint(
    student_ckpt, device,
    num_output_tokens=NUM_OUTPUT_TOKENS,
    embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS,
)
result = evaluate_teacher(
    student, test_loader, device, ds.class_names,
    results_dir=f"{PROJECT_DIR}/results",
    results_subdir=f"student_K{NUM_OUTPUT_TOKENS}",
)
print("Student test accuracy:", f"{result['test_acc']:.4f}")

In [ ]:
# Save student checkpoint to Drive (skip pushing to GitHub due to size); push logs and results
import shutil

# 1) Copy best student checkpoint to your Drive (so you keep it without pushing to repo)
DRIVE_CKPT_DIR = "/content/drive/MyDrive/Secure-Inference-Token-Reduced-VIT/checkpoints"
Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
src = f"{PROJECT_DIR}/checkpoints/student_best.pt"
dst = f"{DRIVE_CKPT_DIR}/student_best.pt"
if Path(src).exists():
    shutil.copy2(src, dst)
    print("Student checkpoint saved to Drive:", dst)
else:
    print("No student_best.pt found at", src)

# 2) Push only logs and results to GitHub (no checkpoints)
GITHUB_TOKEN = "your_token_here"
%cd $PROJECT_DIR
!git config --global user.name "PulockDas"
!git config --global user.email "pulockkamol50@gmail.com"
!git add results/ logs/
!git commit -m "Add student distillation logs and evaluation results" || echo "No changes to commit"
# Uncomment below and set your token to push:
# !git push https://{GITHUB_TOKEN}@github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git